# Database RAG Test Notebook
## RAG: Build an Evidence Pipeline from the Medication Database

**Central question:** What changes when an answer must show database evidence?

This notebook builds a small, inspectable RAG pipeline using **TF-IDF + cosine similarity** over records in the project SQLite database. It uses no API key, external service, or fabricated source records. The goal is to see how chunking, query wording, and top-k affect which database evidence becomes visible.

The source of truth is `medications.db`: drug interactions and non-obsolete Human Phenotype Ontology (HPO) terms. The notebook reads the database in read-only mode.

## What to save as evidence

For each experiment, record:

| Change made | Retrieved evidence | Answer / citation | Failure label | Interpretation |
|---|---|---|---|---|
| chunking / query / top-k | source IDs + scores | supported or unsupported? | R1 / R2 / R3 / G1 / C1 / E1 | what the evidence means |

A justified **reject** or **defer** decision is valid evidence. R1 = retrieval miss, R2 = chunk-boundary failure, R3 = irrelevant evidence in context, G1 = unsupported generation, C1 = citation mismatch, E1 = evidence is insufficient.

In [1]:
import math
import re
import sqlite3
from collections import Counter, defaultdict
from pathlib import Path

TOKEN_PATTERN = re.compile(r'[a-z0-9]+', re.IGNORECASE)
REQUIRED_TABLES = {'drugs', 'interactions', 'hpo_terms', 'hpo_synonyms', 'hpo_edges'}


## Source corpus — load records from SQLite

The database is searched directly; the JSON source files are not separately indexed. Each document keeps a stable database ID so provenance survives chunking and retrieval. If the notebook cannot find a populated database in its working directory or a parent directory, stop and load the project data first.

In [2]:
working_dir = Path.cwd().resolve()
db_candidates = [working_dir / 'medications.db'] + [parent / 'medications.db' for parent in working_dir.parents]
DB_PATH = next((path for path in db_candidates if path.is_file()), None)
if DB_PATH is None:
    raise FileNotFoundError('Could not find medications.db in the notebook folder or its parent folders.')

connection = sqlite3.connect(DB_PATH.resolve().as_uri() + '?mode=ro', uri=True)
try:
    tables = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type = 'table'")}
    missing_tables = REQUIRED_TABLES - tables
    if missing_tables:
        raise RuntimeError(f'Database is missing required tables: {sorted(missing_tables)}. Load the project database first.')

    interaction_rows = connection.execute('''
        SELECT i.interaction_id, i.severity, a.name, b.name,
               i.mechanism, i.effect, i.safer_alternative, i.rationale, i.reference
        FROM interactions AS i
        JOIN drugs AS a ON a.drug_id = i.drug_a_id
        JOIN drugs AS b ON b.drug_id = i.drug_b_id
        ORDER BY i.interaction_id
    ''').fetchall()
    phenotype_rows = connection.execute('''
        SELECT t.hp_id, t.label, t.definition, t.comment,
               GROUP_CONCAT(s.synonym, ', ')
        FROM hpo_terms AS t
        LEFT JOIN hpo_synonyms AS s ON s.hp_id = t.hp_id
        WHERE t.is_obsolete = 0
        GROUP BY t.hp_id, t.label, t.definition, t.comment
        ORDER BY t.hp_id
    ''').fetchall()
finally:
    connection.close()

documents = []
for row in interaction_rows:
    record_id, severity, drug_a, drug_b, mechanism, effect, alternative, rationale, reference = row
    fields = [
        ('Severity', severity), ('Drug A', drug_a), ('Drug B', drug_b),
        ('Mechanism', mechanism), ('Effect', effect),
        ('Safer alternative', alternative), ('Rationale', rationale), ('Reference', reference),
    ]
    text = '\n'.join(f'{label}: {value}' for label, value in fields if value is not None and str(value).strip())
    documents.append({'source_id': f'ddi:{record_id}', 'category': 'drug interaction',
                      'title': f'{drug_a} + {drug_b} ({severity})', 'text': text})

for hp_id, label, definition, comment, synonyms in phenotype_rows:
    fields = [('HPO ID', hp_id), ('Label', label), ('Definition', definition),
              ('Comment', comment), ('Synonyms', synonyms)]
    text = '\n'.join(f'{field}: {value}' for field, value in fields if value is not None and str(value).strip())
    documents.append({'source_id': f'hpo:{hp_id}', 'category': 'HPO term',
                      'title': label, 'text': text})

print(f'Database: {DB_PATH}')
print(f'Loaded {len(interaction_rows):,} interactions and {len(phenotype_rows):,} HPO terms ({len(documents):,} records total).')

Database: C:\Users\skyvu\Documents\CSCI_2600\S26\Med-Interaction-and-information\medications.db
Loaded 180 interactions and 19,894 HPO terms (20,074 records total).


In [3]:
def tokenize(text):
    return TOKEN_PATTERN.findall(text.casefold())

def chunk_documents(docs, chunk_words=26, overlap_words=6):
    if chunk_words < 1 or overlap_words < 0 or overlap_words >= chunk_words:
        raise ValueError('Require chunk_words > 0 and 0 <= overlap_words < chunk_words.')
    chunks = []
    step = chunk_words - overlap_words
    for doc in docs:
        words = doc['text'].split()
        for chunk_number, start in enumerate(range(0, len(words), step), start=1):
            piece = words[start:start + chunk_words]
            if not piece:
                break
            chunks.append({'chunk_id': f"{doc['source_id']}_C{chunk_number}",
                           'source_id': doc['source_id'], 'category': doc['category'],
                           'title': doc['title'], 'text': ' '.join(piece)})
            if start + chunk_words >= len(words):
                break
    return chunks

def build_index(chunks):
    if not chunks:
        raise ValueError('Cannot index an empty set of chunks.')
    term_counts = [Counter(tokenize(chunk['text'])) for chunk in chunks]
    document_frequency = Counter()
    for counts in term_counts:
        document_frequency.update(counts.keys())
    total_documents = len(chunks)
    inverse_document_frequency = {
        term: math.log((1 + total_documents) / (1 + frequency)) + 1
        for term, frequency in document_frequency.items()
    }
    postings = defaultdict(list)
    for index, counts in enumerate(term_counts):
        weights = {term: (1 + math.log(count)) * inverse_document_frequency[term]
                   for term, count in counts.items()}
        norm = math.sqrt(sum(weight * weight for weight in weights.values()))
        if norm:
            for term, weight in weights.items():
                postings[term].append((index, weight / norm))
    return postings, inverse_document_frequency

def retrieve(query, chunks, index, top_k=3):
    if top_k < 1:
        raise ValueError('top_k must be greater than zero.')
    postings, inverse_document_frequency = index
    counts = Counter(tokenize(query))
    query_weights = {term: (1 + math.log(count)) * inverse_document_frequency[term]
                     for term, count in counts.items() if term in inverse_document_frequency}
    norm = math.sqrt(sum(weight * weight for weight in query_weights.values()))
    if not norm:
        return []
    scores = defaultdict(float)
    for term, weight in query_weights.items():
        for chunk_index, document_weight in postings[term]:
            scores[chunk_index] += (weight / norm) * document_weight
    ranked = sorted(scores.items(), key=lambda item: (-item[1], chunks[item[0]]['source_id'], chunks[item[0]]['chunk_id']))
    results = []
    for chunk_index, score in ranked[:top_k]:
        results.append({**chunks[chunk_index], 'score': score})
    return results

def extractive_answer(retrieved):
    if not retrieved:
        return 'No matching database evidence was retrieved.'
    best = retrieved[0]
    sentences = [part.strip() for part in re.split(r'(?<=[.!?])\s+', best['text']) if part.strip()]
    evidence = sentences[0] if sentences else best['text']
    return f"{evidence} [{best['chunk_id']}]"

def run_rag(query, docs, chunk_words=26, overlap_words=6, top_k=3):
    chunks = chunk_documents(docs, chunk_words, overlap_words)
    index = build_index(chunks)
    retrieved = retrieve(query, chunks, index, top_k)
    answer = extractive_answer(retrieved)
    return chunks, retrieved, answer

def show_retrieved(retrieved):
    for row in retrieved:
        print(f"{row['chunk_id']} | score={row['score']:.4f} | {row['title']}")
        print(f"  {row['text'][:500]}")
    if not retrieved:
        print('No matching records.')

## Baseline

Ask a question whose answer exists in the database. Inspect **which chunks** become visible to the answer; do not evaluate only the final sentence. The extractive answer quotes retrieved database text and cites its chunk ID.

In [4]:
query = 'Do cabergoline and rucaparib interact, and how severe is it?'
chunks, retrieved, answer = run_rag(query, documents, chunk_words=26, overlap_words=6, top_k=3)
print('ANSWER:', answer)
show_retrieved(retrieved)

ANSWER: how the lesion appears or is composed. [hpo:HP:0020010_C2]
hpo:HP:0020010_C2 | score=0.3066 | Cutaneous lesion morphology
  how the lesion appears or is composed.
hpo:HP:0033070_C2 | score=0.2559 | Impaired self monitoring
  on others and how it compares with standards or expectations for behavior.
hpo:HP:6001533_C5 | score=0.2503 | Abnormal body image
  Body image is how an individual considers or imagines their body and how it appears to others. Gender and social standards might influence the ideal physique.


## Experiment 1 — Change chunking

Change **one variable**. Compare the baseline with `chunk_words=10` and `overlap_words=0`.

Questions:
- Did the drug names and interaction evidence split across chunks?
- Did a needed condition or field disappear from the top-ranked chunk?
- Is this a retrieval miss (R1) or a chunk-boundary failure (R2)?

In [5]:
chunks_small, retrieved_small, answer_small = run_rag(
    query, documents, chunk_words=10, overlap_words=0, top_k=3
)
print('ANSWER:', answer_small)
show_retrieved(retrieved_small)

ANSWER: body and how it appears to others. [hpo:HP:6001533_C10]
hpo:HP:6001533_C10 | score=0.3506 | Abnormal body image
  body and how it appears to others. Gender and social
hpo:HP:0011270_C3 | score=0.3243 | Duplicated tragus
  the normal tragus. Comment: It is unclear how often, or
hpo:HP:5200003_C2 | score=0.3240 | Excessive interest in others
  strong desire to interact and engage with others that is


## Experiment 2 — Change query wording and top-k

Try `top_k=1` and `top_k=4`. Then rewrite the query using different language. Compare the retrieved IDs and scores.

Ask: does more context help, or does it add distraction?

In [ ]:
query2 = 'Does taking clopidogrel with omeprazole reduce the antiplatelet effect?'
chunks2, retrieved2, answer2 = run_rag(query2, documents, chunk_words=26, overlap_words=6, top_k=1)
print('ANSWER:', answer2)
show_retrieved(retrieved2)

_, retrieved2_more, _ = run_rag(query2, documents, chunk_words=26, overlap_words=6, top_k=4)
print('\nSame query with top_k=4:')
show_retrieved(retrieved2_more)

ANSWER: Severity: Major Drug A: Clopidogrel Drug B: Omeprazole Mechanism: CYP2C19 inhibition → decreased clopidogrel efficacy Effect: Reduced antiplatelet effect, increased thrombosis risk Safer alternative: Pantoprazole or [ddi:4_C1]
ddi:4_C1 | score=0.5090 | Clopidogrel + Omeprazole (Major)
  Severity: Major Drug A: Clopidogrel Drug B: Omeprazole Mechanism: CYP2C19 inhibition → decreased clopidogrel efficacy Effect: Reduced antiplatelet effect, increased thrombosis risk Safer alternative: Pantoprazole or


## Experiment 3 — Inspect natural distractors

The full database is the only source; no synthetic or non-authoritative medical record is added. A broad query can still retrieve records that contain overlapping words but do not answer the same question. Inspect the larger context and label irrelevant evidence **R3**.

In [ ]:
broad_query = 'bleeding risk'
chunks3, retrieved3, answer3 = run_rag(broad_query, documents, chunk_words=26, overlap_words=6, top_k=8)
print('ANSWER:', answer3)
show_retrieved(retrieved3)

ANSWER: Bleeding with minor or no trauma, Easy bleeding [hpo:HP:0011889_C2]
hpo:HP:0011889_C2 | score=0.4322 | Bleeding with minor or no trauma
  Bleeding with minor or no trauma, Easy bleeding
ddi:1_C1 | score=0.4201 | Warfarin + Ibuprofen (Major)
  Severity: Major Drug A: Warfarin Drug B: Ibuprofen Mechanism: Increased bleeding risk (antiplatelet effect, GI irritation) Effect: Increased risk of serious GI bleeding Safer alternative: Acetaminophen
hpo:HP:0030747_C3 | score=0.3834 | Preterm intraventricular hemorrhage
  10 weeks early are at highest risk for this type of bleeding. Synonyms: Preterm intraventricular haemorrhage
hpo:HP:0001892_C1 | score=0.3755 | Abnormal bleeding
  HPO ID: HP:0001892 Label: Abnormal bleeding Definition: An abnormal susceptibility to bleeding, often referred to as a bleeding diathesis. A bleeding diathesis may be related to
hpo:HP:0030140_C1 | score=0.3752 | Oral cavity bleeding
  HPO ID: HP:0030140 Label: Oral cavity bleeding Definition: Recurrent or ex

## Evidence check — relevance, support, citation alignment

For each answer, check three things separately:
1. **Relevance:** are the retrieved chunks about the question?
2. **Support:** does the cited text justify the answer claim?
3. **Citation alignment:** does the cited chunk ID point to the displayed supporting database text?

The notebook does not generate new clinical claims. Treat retrieval as an experiment, not as medical advice.

In [ ]:
evidence = [
    {'change_made': 'baseline', 'retrieved_evidence': ', '.join(row['chunk_id'] for row in retrieved),
     'answer_citation': answer, 'failure_label': 'TBD', 'interpretation': 'TBD'},
    {'change_made': 'smaller chunks / no overlap', 'retrieved_evidence': ', '.join(row['chunk_id'] for row in retrieved_small),
     'answer_citation': answer_small, 'failure_label': 'TBD', 'interpretation': 'TBD'},
    {'change_made': 'paraphrased query / top_k=1', 'retrieved_evidence': ', '.join(row['chunk_id'] for row in retrieved2),
     'answer_citation': answer2, 'failure_label': 'TBD', 'interpretation': 'TBD'},
    {'change_made': 'broad query / database distractors', 'retrieved_evidence': ', '.join(row['chunk_id'] for row in retrieved3),
     'answer_citation': answer3, 'failure_label': 'TBD', 'interpretation': 'TBD'},
]
for row in evidence:
    print(row)

{'change_made': 'baseline', 'retrieved_evidence': 'hpo:HP:0020010_C2, ddi:1_C1, ddi:20_C1', 'answer_citation': 'how the lesion appears or is composed. [hpo:HP:0020010_C2]', 'failure_label': 'TBD', 'interpretation': 'TBD'}
{'change_made': 'smaller chunks / no overlap', 'retrieved_evidence': 'ddi:1_C1, hpo:HP:6001533_C10, hpo:HP:0011270_C3', 'answer_citation': 'Severity: Major Drug A: Warfarin Drug B: Ibuprofen Mechanism: Increased [ddi:1_C1]', 'failure_label': 'TBD', 'interpretation': 'TBD'}
{'change_made': 'paraphrased query / top_k=1', 'retrieved_evidence': 'ddi:4_C1', 'answer_citation': 'Severity: Major Drug A: Clopidogrel Drug B: Omeprazole Mechanism: CYP2C19 inhibition → decreased clopidogrel efficacy Effect: Reduced antiplatelet effect, increased thrombosis risk Safer alternative: Pantoprazole or [ddi:4_C1]', 'failure_label': 'TBD', 'interpretation': 'TBD'}
{'change_made': 'broad query / database distractors', 'retrieved_evidence': 'hpo:HP:0011889_C2, ddi:1_C1, hpo:HP:0030747_C3, 

## Project transfer

Before you leave, answer:
1. What knowledge does your project need, and how often does it change?
2. Which database sources are trustworthy enough to retrieve from?
3. Which retrieval variable would you test first?
4. What would count as success? What would count as failure?
5. Based on today, should your team **adopt**, **modify**, **reject**, or **defer** RAG? Why?

In [ ]:
project_plan = {
    'knowledge_need': '',
    'sources': 'medications.db: drug interactions and HPO terms',
    'retrieval_variable': 'chunking / query wording / top-k / metadata',
    'success_criterion': '',
    'failure_criterion': '',
    'decision': 'adopt / modify / reject / defer',
    'why': '',
}
project_plan

{'knowledge_need': '',
 'sources': 'medications.db: drug interactions and HPO terms',
 'retrieval_variable': 'chunking / query wording / top-k / metadata',
 'success_criterion': '',
 'failure_criterion': '',
 'decision': 'adopt / modify / reject / defer',
 'why': ''}